In [2]:
import os
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from PIL import Image

In [3]:
class imageProcessor :
    def __init__(self,root_dir_path, transformations = None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)
    
    def __getitem__(self,idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = transformations(img)
            return img

In [4]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),#178x218 => 178x178
    transforms.Resize(64), #64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # pixel range : [-1,1]
])

In [5]:
dataset = imageProcessor(root_dir_path, transformations)
print(f"loaded{len(dataset)}images")

loaded202599images


In [6]:
dataLoader = DataLoader(dataset, batch_size=128, shuffle=True)

## Generator network

In [7]:
import torch.nn as nn 
import torch.optim as optim
import numpy as np

In [ ]:
class Genrator(nn.Module):
    def __init__(self,z_dim=100, channels=3):
        super(Genrator,self).__init__

        self.model = nn.Sequential(
            nn.Linear(z_dim,256), # 100 => 256
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024, 64*64*channels),
            nn.Tanh() #[1,-1]
        )
    
    def Forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0),3,64,64)
        return img


## Discriminator Network

In [9]:
class Discriminator(nn.Module):
    def __init__(self, channels=3):
        super(Discriminator,self).__init__

        self.model = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64*64*3,1024), # 100 => 256
            nn.LeakyReLU(),

            nn.Linear(1024,512),
            nn.LeakyReLU(),

            nn.Linear(512,256),
            nn.LeakyReLU(),

            nn.Linear(256,1),
            nn.Sigmoid() #[1,-1]
        )
    
    def Forward(self, img):
        return self.model(img)